In [ ]:
from src.main import MyLinearRegression
from matplotlib import pyplot as plt
import pandas as pd
import json

from src.loss import mae_, rmse_, r2score_

In [ ]:
DATA = "data.csv"
OUT = "thetas.json"

df = pd.read_csv(DATA)
print(df.head())

In [ ]:
df = pd.read_csv(DATA)
x = df["km"].to_numpy(dtype=float).reshape(-1, 1)
y = df["price"].to_numpy(dtype=float).reshape(-1, 1)

mean, std = x.mean(), x.std()
x_n = (x - mean) / std

lr = MyLinearRegression(alpha=0.01, max_iter=2000).fit_(x_n, y)
lr.plot_learning_curve()
t0_n, t1_n = lr.thetas.flatten()

theta1 = t1_n / std
theta0 = t0_n - t1_n * mean / std

with open(OUT, "w") as f:
    json.dump({"theta0": theta0, "theta1": theta1}, f)
print(f"theta0 = {theta0:.4f}, theta1 = {theta1:.6f} -> {OUT}")

y_hat = theta0 + theta1 * x
print(f"R2 = {r2score_(y, y_hat):.4f}")

In [ ]:
try:
    with open("thetas.json") as f:
        t = json.load(f)
    theta0, theta1 = t["theta0"], t["theta1"]
except (FileNotFoundError, KeyError, json.JSONDecodeError):
    theta0, theta1 = 0.0, 0.0

mileage = 70000

print(f"Prix estimé : {(theta0 + theta1 * mileage):.2f}")

In [ ]:
df = pd.read_csv(DATA)
with open("thetas.json") as f:
    t = json.load(f)

y = df["price"].to_numpy(dtype=float)
y_hat = t["theta0"] + t["theta1"] * df["km"].to_numpy(dtype=float)
    
print(f"MAE  : {mae_(y, y_hat):.2f}")
print(f"RMSE : {rmse_(y, y_hat):.2f}")
print(f"R2   : {r2score_(y, y_hat):.4f}")

In [ ]:
df = pd.read_csv(DATA)
with open("thetas.json") as f:
    t = json.load(f)

xs = [df["km"].min(), df["km"].max()]
ys = [t["theta0"] + t["theta1"] * x for x in xs]

plt.scatter(df["km"], df["price"], label="données")
plt.plot(xs, ys, color="red", label="régression linéaire")
plt.xlabel("Kilométrage (km)")
plt.ylabel("Prix")
plt.legend()

# plt.savefig("regression.png", dpi=120)
# print("regression.png enregistré")